# Extended cohort pull

Pulls extended publication histories (2015-2025) for the 10 seed authors from OpenAlex, then fetches each unique co-author's OpenAlex topics/h-index profile.

**Inputs:**
- `seed_authors_for_matching.csv` (10 seed authors with OpenAlex IDs)

**Outputs:**
- `extended_cohort_raw.csv` — one row per (paper, author, affiliation) for seed publications
- `author_topics.json` — OpenAlex Author.topics profile + h-index for every author in the extended cohort
- `adapted_extended_cohort.csv` — same as raw but with columns renamed for scoring pipeline and per-author topics applied

**Notes on decisions made:**
- Jianjun Zhang's initial OpenAlex ID was a name-conflation of multiple researchers. Corrected via DOI-based verification to `A5100358209` (China University of Geosciences materials scientist).
- Multi-affiliation authors are preserved (multiple rows per author-paper) for downstream weighted country handling.
- Authors without an OpenAlex.topics profile fall back to the paper's primary topic.
- Rows missing country or specialization data are dropped in the analysis pipeline (~20% loss, primarily older publications with incomplete OpenAlex metadata).

## Setup

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
from pyalex import Authors, Works, config

config.email = "arav_kumar@berkeley.edu"  

## Part 1: Fetch seed authors' publication histories

For each of the 10 seeds, query all their publications 2015-2025, flatten to one row per (paper, author, affiliation).

In [2]:
def fetch_seed_publications(openalex_id, from_year=2015, to_year=2025):
    """Return all publications by one seed author, flattened to one row per (paper, author)."""
    rows = []
    works = Works().filter(
        author={"id": f"https://openalex.org/{openalex_id}"},
        publication_year=f"{from_year}-{to_year}"
    )

    for page in works.paginate(per_page=200, n_max=None):
        for w in page:
            primary_topic = w.get("primary_topic") or {}

            for authorship in w["authorships"]:
                author = authorship["author"]
                if not author.get("id"):
                    continue

                institutions = authorship["institutions"]
                if not institutions:
                    institutions = [{}]

                for inst in institutions:
                    rows.append({
                        "pub_id": w["id"].replace("https://openalex.org/", ""),
                        "doi": (w.get("doi") or "").replace("https://doi.org/", ""),
                        "pub_date": w.get("publication_date"),
                        "author_openalex_id": author["id"].replace("https://openalex.org/", ""),
                        "display_name": author["display_name"],
                        "aff_country_code": inst.get("country_code"),
                        "aff_name": inst.get("display_name"),
                        "subfield": (primary_topic.get("subfield") or {}).get("display_name"),
                        "field": (primary_topic.get("field") or {}).get("display_name"),
                        "domain": (primary_topic.get("domain") or {}).get("display_name"),
                        "citations": w.get("cited_by_count"),
                    })

    return pd.DataFrame(rows)

In [ ]:
seeds = pd.read_csv("seed_authors_for_matching.csv")
print(f"Loaded {len(seeds)} seed authors")

all_frames = []
for i, row in seeds.iterrows():
    oid = row["openalex_id"]
    seed_label = row["author_anon_id"]
    print(f"[{i+1}/{len(seeds)}] Fetching {seed_label} ({row['first_name']} {row['last_name']}, {oid})...")
    df = fetch_seed_publications(oid, 2015, 2025)
    df["seed_author_anon_id"] = seed_label
    all_frames.append(df)
    print(f"  -> {len(df):,} rows")

combined = pd.concat(all_frames, ignore_index=True)

# Deduplicate (papers co-authored by multiple seeds appear once per seed's query)
combined = combined.drop_duplicates(
    subset=["pub_id", "author_openalex_id", "aff_country_code"],
    keep="first"
).reset_index(drop=True)

combined.to_csv("extended_cohort_raw.csv", index=False)
print(f"\nSaved extended_cohort_raw.csv: {len(combined):,} rows, "
      f"{combined['pub_id'].nunique():,} papers, "
      f"{combined['author_openalex_id'].nunique():,} authors")

## Part 2: Fetch each unique author's OpenAlex topics + h-index

Batched to avoid rate limits. Resumes from `author_topics.json` if partially complete.

In [ ]:
def fetch_author_batch(openalex_ids):
    """Fetch up to 50 authors in one query."""
    id_filter = "|".join(openalex_ids)
    return Authors().filter(openalex_id=id_filter).get(per_page=len(openalex_ids))


def extract_author_metadata(author_obj):
    """Extract top topic + h-index from one Author response."""
    topics = author_obj.get("topics") or []
    top = topics[0] if topics else None
    return {
        "subfield": (top.get("subfield") or {}).get("display_name") if top else None,
        "field": (top.get("field") or {}).get("display_name") if top else None,
        "domain": (top.get("domain") or {}).get("display_name") if top else None,
        "h_index": (author_obj.get("summary_stats") or {}).get("h_index"),
        "works_count": author_obj.get("works_count"),
        "cited_by_count": author_obj.get("cited_by_count"),
    }


def fetch_all_authors_batched(ids_to_fetch, existing=None, batch_size=50, sleep_between=0.2):
    """Batched fetch with resume support. Saves checkpoint every 20 batches."""
    results = dict(existing) if existing else {}
    output_path = Path("author_topics.json")
    total = len(ids_to_fetch)

    for i in range(0, total, batch_size):
        batch = ids_to_fetch[i:i+batch_size]
        try:
            authors = fetch_author_batch(batch)
            fetched_ids = set()
            for a in authors:
                aid = a["id"].replace("https://openalex.org/", "")
                results[aid] = extract_author_metadata(a)
                fetched_ids.add(aid)
            # Mark batch members not returned as null (missing from OpenAlex)
            for aid in batch:
                if aid not in fetched_ids and aid not in results:
                    results[aid] = None
        except Exception as e:
            print(f"  batch {i//batch_size} failed: {e}")

        if (i // batch_size) % 20 == 0:
            with open(output_path, "w") as f:
                json.dump({k: v for k, v in results.items() if v is not None}, f, indent=2)
            print(f"  [{i:5}/{total}] checkpoint ({sum(1 for v in results.values() if v is not None):,} entries)")

        time.sleep(sleep_between)

    with open(output_path, "w") as f:
        json.dump({k: v for k, v in results.items() if v is not None}, f, indent=2)
    print(f"Done. {sum(1 for v in results.values() if v is not None):,} authors with metadata")
    return results

In [ ]:
# Resume from existing if present
try:
    with open("author_topics.json") as f:
        existing_topics = json.load(f)
    print(f"Resuming from {len(existing_topics):,} existing entries")
except FileNotFoundError:
    existing_topics = {}
    print("Starting fresh")

combined = pd.read_csv("extended_cohort_raw.csv")
all_authors = combined["author_openalex_id"].unique().tolist()
to_fetch = [aid for aid in all_authors if aid not in existing_topics]
print(f"Total authors: {len(all_authors):,}, need to fetch: {len(to_fetch):,}")

if to_fetch:
    author_topics = fetch_all_authors_batched(to_fetch, existing=existing_topics)
else:
    author_topics = existing_topics
    print("Nothing to fetch")

## Part 3: Adapt for scoring pipeline

Renames columns to match existing `run_analysis.py`, applies per-author topics, adds `gender="unknown"` for compatibility.

In [3]:
from commindex import make_topic_key


def adapt_extended_for_pipeline(df, author_topics_path="author_topics.json"):
    """Rename OpenAlex columns to match existing scoring pipeline conventions.
    Assigns each author their own OpenAlex topic; falls back to paper's topic."""
    with open(author_topics_path) as f:
        author_topics = json.load(f)

    df = df.copy()

    def get_specialization(row):
        aid = row["author_openalex_id"]
        topic = author_topics.get(aid)
        if topic is not None:
            return make_topic_key(topic["subfield"], topic["field"], topic["domain"])
        # Fallback to paper's topic
        if pd.notna(row["subfield"]):
            return make_topic_key(row["subfield"], row["field"], row["domain"])
        return None

    df["specialization"] = df.apply(get_specialization, axis=1)

    df = df.rename(columns={
        "pub_id": "pub_anon_id",
        "author_openalex_id": "author_anon_id",
    })
    df["gender"] = "unknown"
    return df


combined = pd.read_csv("extended_cohort_raw.csv")
adapted = adapt_extended_for_pipeline(combined)
adapted.to_csv("adapted_extended_cohort.csv", index=False)
print(f"Saved adapted_extended_cohort.csv: {len(adapted):,} rows")

Saved adapted_extended_cohort.csv: 22,261 rows


## Part 4: Sanity checks

Coverage checks before feeding data to the scoring pipeline.

In [4]:
adapted = pd.read_csv("adapted_extended_cohort.csv", parse_dates=["pub_date"])

# Basic shape
print(f"Rows: {len(adapted):,}")
print(f"Unique papers: {adapted['pub_anon_id'].nunique():,}")
print(f"Unique authors: {adapted['author_anon_id'].nunique():,}")

# Missingness
print(f"\nMissing country: {adapted['aff_country_code'].isna().sum():,} rows")
print(f"Missing specialization: {adapted['specialization'].isna().sum():,} rows")

# Country lookup coverage
from commindex import INCOME_GROUP_LOOKUP
countries_in_data = set(adapted["aff_country_code"].dropna().unique())
missing = countries_in_data - set(INCOME_GROUP_LOOKUP.keys())
if missing:
    print(f"\nWARNING: {len(missing)} country codes not in INCOME_GROUP_LOOKUP: {missing}")
else:
    print(f"\nAll {len(countries_in_data)} country codes covered by lookup")

Rows: 22,261
Unique papers: 2,549
Unique authors: 6,722

Missing country: 4,409 rows
Missing specialization: 0 rows

All 81 country codes covered by lookup


In [5]:
# Per-seed paper retention after dropping missing-data rows
adapted_clean = adapted.dropna(subset=["aff_country_code", "specialization"])

seeds = pd.read_csv("seed_authors_for_matching.csv")
print("Per-seed paper retention after cleanup:")
for _, row in seeds.iterrows():
    oid = row["openalex_id"]
    n_before = adapted[adapted["author_anon_id"] == oid]["pub_anon_id"].nunique()
    n_after = adapted_clean[adapted_clean["author_anon_id"] == oid]["pub_anon_id"].nunique()
    pct = 100 * n_after / n_before if n_before else 0
    marker = "" if pct >= 80 else "  <- low retention"
    print(f"  {row['first_name']} {row['last_name']}: {n_after}/{n_before} papers ({pct:.0f}%){marker}")

Per-seed paper retention after cleanup:
  Hermenegildo García: 492/565 papers (87%)
  Jiaguo Yu: 615/660 papers (93%)
  Bicheng Zhu: 159/162 papers (98%)
  Liuyang Zhang: 175/197 papers (89%)
  Jianjun Zhang: 72/76 papers (95%)
  Chuanjia Jiang: 105/110 papers (95%)
  Junwei Fu: 154/156 papers (99%)
  Liyuan Chai: 341/363 papers (94%)
  Johannes F E Mann: 126/179 papers (70%)  <- low retention
  Rajiv Agarwal: 242/300 papers (81%)


In [15]:
# Sanity check: has the per-author topic assignment produced within-paper variation?
# Sample the largest paper by author count
sample_paper = adapted["pub_anon_id"].value_counts().index[0]
sample = adapted[adapted["pub_anon_id"] == sample_paper]
n_unique_specs = sample["specialization"].nunique()
print(f"Sample paper {sample_paper}: {len(sample)} rows, {n_unique_specs} unique specializations")
print(f"Top 5 specializations on this paper:")
print(sample["specialization"].value_counts().head())

Sample paper W3200915536: 108 rows, 31 unique specializations
Top 5 specializations on this paper:
specialization
Health Sciences||Medicine||Endocrinology, Diabetes and Metabolism    33
Health Sciences||Medicine||Nephrology                                22
Health Sciences||Medicine||Cardiology and Cardiovascular Medicine     4
Health Sciences||Medicine||Surgery                                    4
Health Sciences||Medicine||Pulmonary and Respiratory Medicine         4
Name: count, dtype: int64
